# Day 11 — Logging + monitoring

Add CSV logging to `app.py` and summarize logs.

In [ ]:
!pip -q install pandas

In [ ]:
# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, time, uuid
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
print("BASE:", BASE)


In [ ]:
app_path = f"{BASE}/app.py"
txt = open(app_path,"r",encoding="utf-8").read()

if "LOG_PATH" not in txt:
    inject = '\nLOG_DIR = os.environ.get("LOG_DIR", os.path.join(BASE, "logs"))\nos.makedirs(LOG_DIR, exist_ok=True)\nLOG_PATH = os.path.join(LOG_DIR, "requests.csv")\n\ndef log_request(row):\n    import csv\n    header=["ts","request_id","status","confidence","latency_ms","top_study_ids"]\n    exists=os.path.exists(LOG_PATH)\n    with open(LOG_PATH,"a",newline="") as f:\n        w=csv.DictWriter(f, fieldnames=header)\n        if not exists: w.writeheader()\n        w.writerow(row)\n'
    txt = txt.replace('TOPK = int(os.environ.get("TOPK", "3"))', 'TOPK = int(os.environ.get("TOPK", "3"))' + inject)

if "log_request({" not in txt:
    txt = txt.replace("return JSONResponse({", textwrap.dedent("""top_ids = [c["study_id"] for c in cases]
log_request({"ts": time.strftime("%Y-%m-%d %H:%M:%S"), "request_id": request_id, "status": status,
             "confidence": float(best), "latency_ms": int((time.time()-t0)*1000),
             "top_study_ids": "|".join(map(str, top_ids))})
return JSONResponse({"""))
open(app_path,"w",encoding="utf-8").write(txt)
print("Patched:", app_path)


In [ ]:
log_path = f"{BASE}/logs/requests.csv"
if os.path.exists(log_path):
    logs = pd.read_csv(log_path)
    display(logs.tail())
    print("Status counts:\n", logs["status"].value_counts())
    print("Avg latency:", logs["latency_ms"].mean())
    print("Avg confidence:", logs["confidence"].mean())
else:
    print("No logs yet. Run API and call /predict a few times.")
